# 🕵️ Who is bluffing about AI?
### Reading what S&P 500 executives say about AI, with BERTopic, an LLM and a classifier

**Module 2 · extension notebook** (after notebooks 1–3 of session 1)

Since ChatGPT, every earnings call seems to mention AI. An earnings call is the quarterly phone call where a listed
company's management presents results to investors and answers their questions. Some of what they say about AI is
a real result, some is a plan, and some is just excitement.

We take 7,616 passages about AI from 443 companies and do what an analyst at a bank or a consultancy would do:

1. **Explore:** what do they actually talk about, and how does it change? (BERTopic, notebook 3)
2. **Extract:** turn text into a table with an LLM. (New: *structured extraction*)
3. **Check:** can we trust those labels?
4. **Scale:** train a cheap classifier on the LLM's labels. (Notebook 2)
5. **Report:** let the LLM write a short brief, and verify every quote with code.

**You need:** a free API key, stored as a Colab secret (left sidebar, key icon) under the name `GROQ_API_KEY` from
[console.groq.com](https://console.groq.com) or `GEMINI_API_KEY` from
[aistudio.google.com](https://aistudio.google.com). Without a key the notebook still runs: it uses saved results.

Run all cells (Runtime → Run all). Plotting code is folded away (*Show code*). If Colab asks for access to `HF_TOKEN`,
click *Cancel*. If it asks for access to your API key, click *Grant*.

**Contents**
1. The data
2. What do they talk about? BERTopic
3. From text to a table: structured extraction
4. Can we trust the labels?
5. Scaling up: a hype detector
6. An intelligence brief, with checked quotes
7. Takeaways

In [ ]:
# @title Setup: libraries, chart style, API key { display-mode: "form" }
%pip install -q bertopic openai
import io, json, os, re, urllib.request, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import bertopic, sklearn
from typing import Literal, Optional
from pydantic import BaseModel
from openai import OpenAI, BadRequestError
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from hdbscan import HDBSCAN
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
warnings.filterwarnings("ignore")

NAVY, LAV, GREY, ORANGE = "#211A52", "#C2C1CC", "#54616E", "#D2542A"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.edgecolor": GREY, "axes.labelcolor": GREY,
                     "xtick.color": GREY, "ytick.color": GREY, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E6E6EC", "legend.frameon": False})
pd.set_option("display.max_colwidth", 160)

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

# Change PROVIDER to "gemini" to switch. Model names change often: if one stops working, pick a new one from the provider's list.
PROVIDER = "groq"
PROVIDERS = {"groq": ("https://api.groq.com/openai/v1", "qwen/qwen3.8-27b", "GROQ_API_KEY"),
             "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai", "gemini-flash-lite-latest", "GEMINI_API_KEY")}
base_url, MODEL, key_name = PROVIDERS[PROVIDER]
api_key = secret(key_name)
client = OpenAI(base_url=base_url, api_key=api_key) if api_key else None
print("bertopic", bertopic.__version__, "· scikit-learn", sklearn.__version__, "· pandas", pd.__version__)
print(f"LLM: {PROVIDER} / {MODEL}" if client else f"No {key_name} found: using saved LLM results.")

---
# 📞 1 · The data

The source is the Hugging Face dataset
[`kurry/sp500_earnings_transcripts`](https://huggingface.co/datasets/kurry/sp500_earnings_transcripts): 33,362
transcripts, 3.5 GB. We prepared a small slice for you. Every row is one **passage**: the sentences in one speaker's
turn that mention AI, plus one sentence of context on each side, from 2021 onwards. Passages with a question mark are
dropped, because those are mostly analysts asking, not companies claiming. The sector comes from the S&P 500 list.

The embeddings are ready too (`all-MiniLM-L6-v2`, the model from notebooks 1–3), so you don't wait for them.

In [ ]:
BASE = "https://raw.githubusercontent.com/aaubs/ds-master/main/data/earnings_ai/"
def fetch(name):
    with urllib.request.urlopen(BASE + name) as r:
        return io.BytesIO(r.read())

df = pd.read_parquet(fetch("passages.parquet"))
embeddings = np.load(fetch("embeddings.npy")).astype(np.float32)
print(df.shape, embeddings.shape, "· last call:", df["date"].max())
df[["company", "year", "quarter", "sector", "text"]].sample(3, random_state=3)

In [ ]:
# @title Chart: AI passages and companies per year { display-mode: "form" }
by_year = df.groupby("year").agg(passages=("id", "size"), companies=("symbol", "nunique"))
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.bar(by_year.index, by_year["passages"], color=NAVY)
for x, (p, c) in zip(by_year.index, by_year.values):
    ax.text(x, p + 40, f"{c} companies", ha="center", color=GREY, fontsize=9)
ax.set_ylabel("AI passages"); ax.grid(axis="x", visible=False); ax.set_xticks(by_year.index)
plt.show()

ChatGPT came out in November 2022. Look at what happens next. The 2025 bar is short because the dataset ends in spring 2025.

🤔 *Before you scroll on: what do you think the companies outside tech are saying about AI?*

---
# 🗺️ 2 · What do they talk about? BERTopic

The same recipe as notebook 3: embed, reduce with UMAP, cluster with HDBSCAN, describe each cluster with its most
typical words. Words like *AI* appear in every passage, so they say nothing here. We add them to the stopwords, for the
same reason *Aalborg* got a low IDF in notebook 1.

Expect around 60 % of the passages in topic −1. Most AI talk is generic, and that is a finding in itself.

In [ ]:
filler = ["ai", "artificial", "intelligence", "think", "going", "really", "just", "know", "lot", "like", "things", "thing",
          "right", "way", "yes", "want", "great", "good", "very", "quarter", "year", "see", "seeing", "new", "kind", "sort",
          "question", "thanks", "thank", "gen", "generative", "maybe", "don", "ve", "ll", "re", "use", "using", "got", "look",
          "looking", "mean", "said", "say", "thats", "weve", "youre", "im", "theres"]
topic_model = BERTopic(
    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42),
    hdbscan_model=HDBSCAN(min_cluster_size=40, cluster_selection_method="leaf", prediction_data=True),
    vectorizer_model=CountVectorizer(stop_words=list(ENGLISH_STOP_WORDS) + filler, ngram_range=(1, 2), min_df=5),
    representation_model=MaximalMarginalRelevance(diversity=0.3), top_n_words=10)
df["topic"] = topic_model.fit_transform(df["text"].tolist(), embeddings)[0]
print(f"{df['topic'].nunique() - 1} topics · {(df['topic'] == -1).mean():.0%} of passages without a topic")

info = topic_model.get_topic_info()
info["words"] = info["Representation"].str[:5].str.join(", ")
info.loc[info["Topic"] != -1, ["Topic", "Count", "words"]].head(15)

In [ ]:
def find_topic(word):
    """The largest topic whose top words contain this word."""
    hits = info[(info["Topic"] != -1) & info["Representation"].apply(lambda ws: word in " ".join(ws).split())]
    return int(hits.iloc[0]["Topic"]) if len(hits) else None

picks = {w: find_topic(w) for w in ["gas", "nvidia", "memory", "ads", "health"]}
share = df.groupby(["year", "topic"]).size().unstack(fill_value=0).div(df.groupby("year").size(), axis=0) * 100
print(picks)

In [ ]:
# @title Chart: which topics grow? { display-mode: "form" }
fig, ax = plt.subplots(figsize=(8, 4))
colors = {"gas": ORANGE, "nvidia": NAVY, "memory": "#7A73C9", "ads": LAV, "health": GREY}
for word, t in picks.items():
    if t is not None:
        ax.plot(share.index, share[t], marker="o", lw=2.3, color=colors[word], label=word)
ax.set_xticks(share.index); ax.set_ylabel("% of the year's AI passages"); ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.15))
plt.show()

Share of each year's passages, so the short 2025 still compares fairly.

Look for the gas topic: energy and utility companies talk about AI because data centres need electricity. Nobody writes
a keyword rule for that in advance. This is what topic models are good for: finding the question you did not know to ask.

🔧 *Try another word in `find_topic`: `capex`, `security`, `travel`, `adobe`.*

In [ ]:
gas = df[df["topic"] == picks["gas"]]
print(gas["sector"].value_counts().head(4).to_string(), "\n")
for d in topic_model.get_representative_docs(picks["gas"])[:2]:
    print("•", d[:350], "\n")

---
# 🧾 3 · From text to a table: structured extraction

A topic model tells you what passages are about. It cannot tell you whether the company actually achieved anything.
For that we use an LLM, and we ask for a fixed format instead of free text. This is **structured extraction**: you define
the columns, the LLM fills them in for each text. We define the columns with a `pydantic` class, and the API
guarantees that the answer has exactly that shape.

We ask for one `claim_type` per passage:
- `concrete_result`: something already happened, with a specific outcome
- `plan`: an intention, investment or launch without a result yet
- `vague_hype`: excitement with no specific outcome or plan

The instructions are the most important code in this notebook. Read them.

In [ ]:
SYSTEM = """You read passages from company earnings calls in which someone mentions AI.
For each passage, return one record:
- use_case: what the AI is used for, in at most 6 plain words. If no use is named, write "not stated".
- claim_type, exactly one of:
    concrete_result: reports something that has already happened, with a specific outcome
      (revenue, customers, time or cost saved, a product in use by named or counted customers).
    plan: a stated intention, investment, launch or timeline that has not produced a result yet.
    vague_hype: enthusiasm, opportunity or positioning with no specific outcome or plan.
- number: a figure supporting the claim, copied exactly from the passage, or null if there is none.
- evidence: the shortest exact quote from the passage (copied word for word, at most 20 words) that justifies claim_type.
Judge only what the passage says. Excitement is not a result: words like "huge", "strong" or "great response"
without a figure, a named customer or a product in use are vague_hype. Forecasts of a market or a future number are plan
or vague_hype, not concrete_result."""

class Record(BaseModel):
    id: int
    use_case: str
    claim_type: Literal["concrete_result", "plan", "vague_hype"]
    number: Optional[str]
    evidence: str

class Batch(BaseModel):
    records: list[Record]

def strict_schema(model):
    """Pydantic to the strict JSON schema the APIs want: every field required, no extras."""
    s = model.model_json_schema()
    def walk(n):
        if isinstance(n, dict):
            if "properties" in n:
                n["additionalProperties"] = False
                n["required"] = list(n["properties"])
            for v in n.values():
                walk(v)
        elif isinstance(n, list):
            for v in n:
                walk(v)
    walk(s)
    return s

def ask(system, user, model_class):
    extra = {"reasoning_format": "hidden"} if MODEL.startswith("qwen") else {}
    for temperature in (0, 0.3):   # models sometimes return JSON that breaks the schema: try once more
        try:
            r = client.chat.completions.create(
                model=MODEL, temperature=temperature,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
                response_format={"type": "json_schema", "json_schema": {"name": "answer", "strict": True, "schema": strict_schema(model_class)}},
                extra_body=extra)
            return model_class.model_validate_json(r.choices[0].message.content)
        except BadRequestError as e:
            print("Retrying:", str(e)[:110])
    raise e

We send 10 passages per call. Free accounts have limits per minute and per day, so **this notebook makes only a few
calls**. We use 20 passages that also have saved labels, so you can compare your run with the saved one.

In [ ]:
saved = pd.read_json(fetch("llm_labels.json")).merge(df, on="id")
demo = saved.sample(20, random_state=1)

try:
    live = []
    for i in range(0, 20, 10):
        part = demo.iloc[i:i + 10]
        user = "\n\n".join(f"[id {r.id}] {r.company}, {r.year}: {r.text}" for r in part.itertuples())
        live += [r.model_dump() for r in ask(SYSTEM, user, Batch).records]
    live = pd.DataFrame(live)
except Exception as e:   # no key, quota used up, or a malformed answer
    print("Using saved labels:", type(e).__name__, str(e)[:110])
    live = demo[["id", "use_case", "claim_type", "number", "evidence"]]

check = live.merge(demo[["id", "company", "text", "claim_type"]], on="id", suffixes=("", "_saved"))
check["quote_found"] = [e in t for e, t in zip(check["evidence"], check["text"])]
print(f"same label as saved: {(check['claim_type'] == check['claim_type_saved']).mean():.0%} · quotes found in the passage: {check['quote_found'].mean():.0%}")
check[["company", "use_case", "claim_type", "number", "evidence", "quote_found"]]

We just turned free text into a table you can filter, count and plot. And `quote_found` is a check the LLM cannot talk its
way around: plain Python, `quote in text`.

🔧 *Set `PROVIDER = "gemini"` in the setup cell and rerun. Same text, same instructions: does a different model agree?*

---
# 🔍 4 · Can we trust the labels?

We labelled 300 passages in advance. Of those, 239 remain after dropping questions. Before using them, we had an independent
reader label 60 of the 300 by hand, with the same definitions and without seeing the LLM's answers:

| | All 60 passages | Only where the reader was sure (26) |
|---|---|---|
| Same label (3 classes) | 72 % | 81 % |
| Same on hype vs the rest | 80 % | 92 % |

The reader was unsure on 34 of 60. Many passages really are in between: *launched, and performing well*. Is that a result?
When the LLM and the reader disagreed, the LLM was usually too generous and said `concrete_result` where the reader said hype.

So the labels are useful and imperfect, and part of the problem lies in the question, not in the model.

In [ ]:
saved["quote_found"] = [e in t for e, t in zip(saved["evidence"], saved["text"])]
print(saved["claim_type"].value_counts().to_string())
print(f"\nquotes found word for word: {saved['quote_found'].mean():.0%}")
print(f"passages with a number: {saved['number'].notna().mean():.0%}")
print(f"use case not stated: {(saved['use_case'].str.lower() == 'not stated').mean():.0%}")

### ✏️ Try it: be the second reader
Here are six passages. Decide on a label for each one **before** you run the next cell.

In [ ]:
quiz = saved.sample(6, random_state=11)
for i, r in enumerate(quiz.itertuples(), 1):
    print(f"{i}. {r.company}, {r.year}\n   {r.text[:420]}\n")

In [ ]:
quiz[["company", "claim_type", "use_case", "evidence"]].reset_index(drop=True)

---
# 🚨 5 · Scaling up: a hype detector

An LLM per passage is slow and uses up your quota. So we do what notebook 2 did: train a small classifier on the labelled
passages and apply it to all 7,616. It takes seconds and costs nothing.

Three classes turned out to be too hard with 239 labels, so we ask a simpler question: **is this passage vague hype, yes
or no?** Hype is about 23 % of the labels, so always saying "no" already gets about 0.77. That is the number to beat.
We also look at **AUC** from module 1: how well the model ranks hype above the rest. 0.5 is random guessing.

In notebook 2, embeddings beat TF-IDF. Will they win here too?

In [ ]:
y = (saved["claim_type"] == "vague_hype").astype(int).values
cv = StratifiedKFold(5, shuffle=True, random_state=42)
tfidf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                      LogisticRegression(max_iter=3000, class_weight="balanced", C=3))
sbert = LogisticRegression(max_iter=3000, class_weight="balanced", C=3)
X_sbert = embeddings[df.set_index("id").index.get_indexer(saved["id"])]

for name, model, X in [("TF-IDF", tfidf, saved["text"]), ("SBERT", sbert, X_sbert)]:
    p = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]
    print(f"{name:<7} accuracy {accuracy_score(y, p > 0.5):.2f} · AUC {roc_auc_score(y, p):.2f}")

TF-IDF wins this time. Embeddings capture what a text is **about**. Hype is not a topic: it is a **way of talking**, and that
lives in small words that SBERT mostly ignores and TF-IDF counts. The best representation depends on the question.

Accuracy barely beats 0.77, so this is no lie detector. AUC says it is useful for ranking: sorting passages from
most to least hype-like. Let's look at the words.

In [ ]:
tfidf.fit(saved["text"], y)
words = tfidf[0].get_feature_names_out()
order = np.argsort(tfidf[1].coef_[0])
print("sounds like hype:   ", ", ".join(words[order[::-1][:20]]))
print("sounds like results:", ", ".join(words[order[:20]]))

Hype sounds like *I think*, *the most*, *compelling*, *big*, *a lot of*. Results sound like *our*, *quarter*, *products*,
*example*, *sales*. Some of these words will be accidents of 239 labels, so don't quote them as a law of nature.

In the most hype-like list below you will meet a few analysts whose question lost its question mark when we cut the passage.
Real data is like that.

In [ ]:
df["hype_score"] = tfidf.predict_proba(df["text"])[:, 1]
cols = ["company", "year", "hype_score", "text"]
print("Most hype-like:")
display(df.sort_values("hype_score", ascending=False)[cols].head(4))
print("Least hype-like:")
display(df.sort_values("hype_score")[cols].head(4))

In [ ]:
(df.groupby("sector")["hype_score"].agg(["mean", "size"]).sort_values("mean").round(2))

So, who is bluffing? **At sector level, we cannot say.** The averages are close together, the classifier is weak, and
the score is not a percentage of hype. It would be easy to make a dramatic chart from this table, and it would be wrong.

The classifier is good for one thing: sorting thousands of passages so a human reads the most interesting ones first.

🤔 *An analyst wants to use this score to short "AI-washing" companies. What would you tell them?*

---
# 📝 6 · An intelligence brief, with checked quotes

Now the part clients pay for: a one-page brief on a topic. We take the gas and power topic, use the classifier to keep the
passages that sound least like hype, and ask the LLM to write a brief. It has to back up every finding with a word-for-word
quote, and Python checks each quote against the source. A single call.

In [ ]:
class Finding(BaseModel):
    company: str
    finding: str
    quote: str

class Brief(BaseModel):
    headline: str
    findings: list[Finding]
    what_is_missing: str

BRIEF = """You write a short brief for an investment team, using only the passages you are given.
Return one JSON object with exactly these three fields, in this order:
headline: one plain sentence, no hype.
findings: a list of 4 to 6 items, each with company, finding (one sentence) and quote (an exact word-for-word copy from a passage, at most 25 words).
what_is_missing: one or two sentences on what these passages do not tell us."""

topic_id = picks["gas"]
pool = df[df["topic"] == topic_id].sort_values("hype_score").head(25)
source = "\n\n".join(f"{r.company}, {r.year}: {r.text}" for r in pool.itertuples())

try:
    brief = ask(BRIEF, source, Brief)
except Exception as e:   # no key, quota used up, or a malformed answer
    print("Using the saved brief:", type(e).__name__, str(e)[:110], "\n")
    brief = Brief.model_validate_json(fetch("brief.json").read())

print(brief.headline, "\n")
for f in brief.findings:
    ok = any(f.quote in t for t in pool["text"])
    print(f"{'✓' if ok else '✗ NOT FOUND'} {f.company}: {f.finding}\n    \"{f.quote}\"\n")
print("Missing:", brief.what_is_missing)

A ✓ proves the quote exists. It does not prove the finding is a fair reading of it, or that the right company got credit.
That check is yours: pick one finding and read the source passage in `pool`.

🔧 *Change `topic_id` to another topic and generate a new brief. That costs one call.*

---
# 🧰 7 · Takeaways

| Step | Tool | What it gave us | What it cannot do |
|---|---|---|---|
| Explore | BERTopic | themes nobody asked about, like gas and power | tell you whether a claim is true |
| Extract | LLM with a schema | a table from free text | be right every time: about 3 in 4 labels matched a human reader |
| Check | a human, and `quote in text` | a measure of how far to trust it | settle passages that really are in between |
| Scale | TF-IDF classifier | a ranking of 7,616 passages in seconds | convict a sector or a company |
| Report | LLM with quotes | a readable brief with evidence | judge fairness: that's your job |

The pipeline is what a real analyst uses. The cheap methods do the bulk of the work, the LLM does the hard reading on a
sample, and the code checks whatever can be checked.

**Now try it on another question.** Swap the three claim types for your own, for example *cost saving, new revenue, risk*.